# Étape 2c : génération du dictionnaire

Le dictionnaire est proposé à partir des données observées. Toutes les règles sont au statut `observé`, les écarts sont signalés dans `a_arbitrer`, et les colonnes sensibles n'exposent aucune valeur.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

In [ ]:
%%time
from frigeo.chargement import charger_tout
from frigeo.dictionnaire import generer_dictionnaire

tables = charger_tout(perimetre)
dictionnaire = generer_dictionnaire(tables)
print({table: len(colonnes) for table, colonnes in dictionnaire.items()})
print(sum(len(colonnes) for colonnes in dictionnaire.values()), "colonnes")

In [ ]:
from frigeo.dictionnaire import resumer_dictionnaire

resume = resumer_dictionnaire(dictionnaire)
print(resume["liste_fermee"].sum(), "listes fermées proposées sur", len(resume), "colonnes")
display(resume[resume["liste_fermee"]].drop(columns=["motif_liste"]))

In [ ]:
ATTENDUES = {
    ("interventions", "categorie"): {
        "Intervention préventive", "Maintenance", "Dépannage", "Contrôle", "Installation",
    },
    ("interventions", "statut"): {"Réalisée", "Annulée", "Reportée"},
    ("interventions", "facturable"): {"O", "N"},
    ("pieces_interventions", "unite"): {"unité", "kg"},
    ("factures_entetes", "type_piece"): {"Facture", "Avoir"},
    ("factures_entetes", "mode_reglement"): {"Virement", "Chèque", "Prélèvement", "Carte"},
    ("factures_lignes", "code_article"): {
        "MO-STD", "MO-URG", "DEPL", "CTRL-F", "PREV-F", "INST-F", "PIECE", "FLUIDE",
    },
    ("factures_lignes", "unite"): {"heure", "forfait", "unité", "kg"},
    ("achats_entetes", "type_piece"): {"Facture", "Avoir"},
    ("achats_lignes", "unite"): {"unité", "kg", "litre", "forfait"},
    ("clients", "type_commerce"): 8,  # libellés exacts non donnés : comparaison en nombre
    ("clients", "contrat_maintenance"): {"O", "N"},
    ("clients", "statut"): {"Actif", "Inactif"},
    ("fournisseurs", "categorie"): {
        "Pièces frigorifiques", "Fluides", "Carburant", "Frais généraux",
    },
    ("fournisseurs", "delai_paiement"): {"30 j fin de mois", "45 j fin de mois"},
    ("fournisseurs", "statut"): {"Actif", "Inactif"},
    ("catalogue_pieces", "famille"): {
        "Compresseur", "Ventilation", "Régulation", "Joint", "Détente", "Résistance",
        "Sonde", "Fluide",
    },
    ("catalogue_pieces", "unite"): {"unité", "kg"},
    ("intervenants", "poste"): {"Technicien frigoriste", "Chef d'équipe", "Apprenti"},
    ("intervenants", "type_contrat"): {"CDI", "CDD", "Apprentissage"},
    ("intervenants", "motif_sortie"): {
        "Démission", "Fin de CDD", "Rupture conventionnelle", "Retraite",
    },
    ("intervenants", "secteur"): {"27", "76"},
}

lignes = []
for (table, colonne), attendu in ATTENDUES.items():
    valeurs = dictionnaire[table][colonne]["valeurs"]
    ligne = {"table": table, "colonne": colonne}
    if not isinstance(valeurs["regle"], list):
        ligne.update(verdict="non proposée", detail=valeurs["motif"])
    elif isinstance(attendu, int):
        verdict = "identique" if len(valeurs["regle"]) == attendu else "écart"
        ligne.update(verdict=verdict, detail=f"{len(valeurs['regle'])} valeurs pour {attendu} attendues")
    else:
        proposees = set(valeurs["regle"])
        a_arbitrer = {element["valeur"] for element in valeurs["a_arbitrer"]}
        manquantes, en_trop = attendu - proposees, proposees - attendu
        if not manquantes and not en_trop:
            verdict = "identique"
        elif manquantes <= a_arbitrer and not en_trop:
            verdict = "valeur rare à arbitrer"
        else:
            verdict = "écart"
        detail = []
        if manquantes:
            detail.append(f"manquantes {sorted(manquantes)}")
        if en_trop:
            detail.append(f"en trop {sorted(en_trop)}")
        ligne.update(verdict=verdict, detail=" ; ".join(detail))
    lignes.append(ligne)

comparaison = pd.DataFrame(lignes)
display(comparaison["verdict"].value_counts().rename("colonnes").to_frame())
display(comparaison[comparaison["verdict"] != "identique"])

proposees_hors_chapitre = resume[
    resume["liste_fermee"] & ~resume.set_index(["table", "colonne"]).index.isin(list(ATTENDUES))
]
display(proposees_hors_chapitre[["table", "colonne", "nb_valeurs", "nb_a_arbitrer"]])

## Synthèse de la comparaison avec le chapitre 4

Sur 22 colonnes à liste fermée du chapitre 4 : 13 sont retrouvées à l'identique, 2 avec une valeur rare à arbitrer (`Avoir` dans `type_piece` des factures et des achats, soit 0,13 % et 0,27 % des lignes), et 7 ne sont pas proposées (6 faute d'effectif, car `fournisseurs` et `intervenants` ont moins de 200 lignes; 1 colonne sensible, `motif_sortie`). Aucune valeur en trop, aucun libellé divergent. Trois listes sont proposées hors chapitre 4 : `JournalCode` et `JournalLib` (plausibles), et `fournisseur_principal` (faux positif, clé vers le référentiel des fournisseurs, à invalider par le métier).